# Practical Exam: House sales

RealAgents is a real estate company that focuses on selling houses.

RealAgents sells a variety of types of house in one metropolitan area.

Some houses sell slowly and sometimes require lowering the price in order to find a buyer.

In order to stay competitive, RealAgents would like to optimize the listing prices of the houses it is trying to sell.

They want to do this by predicting the sale price of a house given its characteristics.

If they can predict the sale price in advance, they can decrease the time to sale.


## Data

The dataset contains records of previous houses sold in the area.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton'. </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced" (two shared walls), "Semi-detached" (one shared wall), or "Detached" (no shared walls). </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |


# Task 1

The team at RealAgents knows that the city that a property is located in makes a difference to the sale price. 

Unfortuntately they believe that this isn't always recorded in the data. 

Calculate the number of missing values of the `city`. 

 - You should use the data in the file "house_sales.csv". 

 - Your output should be an object `missing_city`, that contains the number of missing values in this column. 

In [1]:
import pandas as pd

df = pd.read_csv('house_sales.csv')

missing_city = (df['city'] == '--').sum()
print(missing_city)

73


# Task 2 

Before you fit any models, you will need to make sure the data is clean. 

The table below shows what the data should look like. 

Create a cleaned version of the dataframe. 

 - You should start with the data in the file "house_sales.csv". 

 - Your output should be a dataframe named `clean_data`. 

 - All column names and values should match the table below.


| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
| house_id    | Nominal. </br> Unique identifier for houses. </br>Missing values not possible. |
| city        | Nominal. </br>The city in which the house is located. One of 'Silvertown', 'Riverford', 'Teasdale' and 'Poppleton' </br>Replace missing values with "Unknown". |
| sale_price  | Discrete. </br>The sale price of the house in whole dollars. Values can be any positive number greater than or equal to zero.</br>Remove missing entries. |
| sale_date   | Discrete. </br>The date of the last sale of the house. </br>Replace missing values with 2023-01-01. |
| months_listed  | Continuous. </br>The number of months the house was listed on the market prior to its last sale, rounded to one decimal place. </br>Replace missing values with mean number of months listed, to one decimal place. |
| bedrooms    | Discrete. </br>The number of bedrooms in the house. Any positive values greater than or equal to zero. </br>Replace missing values with the mean number of bedrooms, rounded to the nearest integer. |
| house_type   | Ordinal. </br>One of "Terraced", "Semi-detached", or "Detached". </br>Replace missing values with the most common house type. |
| area      | Continuous. </br>The area of the house in square meters, rounded to one decimal place. </br>Replace missing values with the mean, to one decimal place. |

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('house_sales.csv')

# city: treat '--' as missing, replace with 'Unknown'
df['city'] = df['city'].replace('--', 'Unknown')

# sale_price: remove rows with missing values
df = df.dropna(subset=['sale_price'])

# sale_date: replace missing with '2023-01-01'
df['sale_date'] = df['sale_date'].fillna('2023-01-01')

# months_listed: replace missing with mean rounded to 1 decimal place
mean_months = round(df['months_listed'].mean(), 1)
df['months_listed'] = df['months_listed'].fillna(mean_months)

# bedrooms: replace missing with mean rounded to nearest integer
mean_bedrooms = int(round(df['bedrooms'].mean()))
df['bedrooms'] = df['bedrooms'].fillna(mean_bedrooms).astype(int)

# house_type: standardize abbreviations, then fill missing with mode
house_type_map = {'Det.': 'Detached', 'Semi': 'Semi-detached', 'Terr.': 'Terraced'}
df['house_type'] = df['house_type'].replace(house_type_map)
df['house_type'] = df['house_type'].fillna(df['house_type'].mode()[0])

# area: strip ' sq.m.' suffix, convert to float, fill missing with mean to 1 decimal
df['area'] = df['area'].str.replace(' sq.m.', '', regex=False).astype(float)
mean_area = round(df['area'].mean(), 1)
df['area'] = df['area'].fillna(mean_area)

clean_data = df
print(clean_data.head())
print(clean_data.dtypes)
print(clean_data.isnull().sum())

   house_id        city  sale_price   sale_date  months_listed  bedrooms  \
0   1217792  Silvertown       55943  2021-09-12            5.4         2   
1   1900913  Silvertown      384677  2021-01-17            6.3         5   
2   1174927   Riverford      281707  2021-11-10            6.9         6   
3   1773666  Silvertown      373251  2020-04-13            6.1         6   
4   1258487  Silvertown      328885  2020-09-24            8.7         5   

      house_type   area  
0  Semi-detached  107.8  
1       Detached  498.8  
2       Detached  542.5  
3       Detached  528.4  
4       Detached  477.1  
house_id           int64
city              object
sale_price         int64
sale_date         object
months_listed    float64
bedrooms           int64
house_type        object
area             float64
dtype: object
house_id         0
city             0
sale_price       0
sale_date        0
months_listed    0
bedrooms         0
house_type       0
area             0
dtype: int64


# Task 3 

The team at RealAgents have told you that they have always believed that the number of bedrooms is the biggest driver of house price. 

Producing a table showing the difference in the average sale price by number of bedrooms along with the variance to investigate this question for the team.

 - You should start with the data in the file 'house_sales.csv'.

 - Your output should be a data frame named `price_by_rooms`. 

 - It should include the three columns `bedrooms`, `avg_price`, `var_price`. 

 - Your answers should be rounded to 1 decimal place.   

In [3]:
import pandas as pd

df = pd.read_csv('house_sales.csv')

price_by_rooms = (
    df.groupby('bedrooms')['sale_price']
    .agg(avg_price='mean', var_price='var')
    .reset_index()
)

price_by_rooms['avg_price'] = price_by_rooms['avg_price'].round(1)
price_by_rooms['var_price'] = price_by_rooms['var_price'].round(1)

print(price_by_rooms)

   bedrooms  avg_price     var_price
0         2    67076.4  5.652896e+08
1         3   154665.1  2.378289e+09
2         4   234704.6  1.725211e+09
3         5   301515.9  2.484328e+09
4         6   375741.3  3.924432e+09


# Task 4

Fit a baseline model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `base_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [4]:
import pandas as pd
from sklearn.linear_model import LinearRegression

train = pd.read_csv('train.csv')
val = pd.read_csv('validation.csv')

features = ['bedrooms', 'months_listed', 'area']

X_train = train[features]
y_train = train['sale_price']
X_val = val[features]

model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_val)

base_result = pd.DataFrame({'house_id': val['house_id'], 'price': predictions})
print(base_result.head())

   house_id          price
0   1331375  118876.064865
1   1630115  257484.218145
2   1645745  383988.545816
3   1336775  117832.366475
4   1888274  256378.708250


# Task 5

Fit a comparison model to predict the sale price of a house.

 1. Fit your model using the data contained in “train.csv” </br></br>

 2. Use “validation.csv” to predict new values based on your model. You must return a dataframe named `compare_result`, that includes `house_id` and `price`. The price column must be your predicted values.

In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder

train = pd.read_csv('train.csv')
val = pd.read_csv('validation.csv')

features = ['city', 'months_listed', 'bedrooms', 'house_type', 'area']
cat_features = ['city', 'house_type']

enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train_enc = train[features].copy()
val_enc = val[features].copy()

train_enc[cat_features] = enc.fit_transform(train[cat_features])
val_enc[cat_features] = enc.transform(val[cat_features])

X_train = train_enc
y_train = train['sale_price']
X_val = val_enc

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_val)

compare_result = pd.DataFrame({'house_id': val['house_id'], 'price': predictions})
print(compare_result.head())

   house_id      price
0   1331375   82256.62
1   1630115  303361.65
2   1645745  404205.67
3   1336775  105696.35
4   1888274  267768.60
